# MIMIC-IV Sepsis Data Preprocessing (Batch Mode)

This notebook preprocesses MIMIC-IV data with **batch processing and checkpoint/resume capability**.

## Key Features:
- ✅ **Batch processing**: Process in chunks of 250 patients
- ✅ **Auto-save**: Progress saved after each batch
- ✅ **Resume capability**: Restart from last checkpoint if interrupted
- ✅ **Memory efficient**: Clears memory between batches

## Processing Pipeline:
1. Load MIMIC-IV data from Google Drive
2. Process patients in batches (250 per batch)
3. Save each batch to separate file
4. Combine all batches at the end
5. Generate Sepsis-3 labels

## Modes:
- **test**: 100 patients (1 batch, ~5 min)
- **medium**: 1,000 patients (4 batches, ~1 hour)
- **large**: 5,000 patients (20 batches, ~5 hours)
- **full**: All ICU patients (~24-48 hours)

**Author**: Jason  
**Date**: January 2026

## 1. Setup & Configuration

In [ ]:
# ========================================
# MODE SELECTION
# ========================================
MODE = "large"  # Options: "test", "medium", "large", "full"
BATCH_SIZE = 250  # Patients per batch (adjust based on memory)

MODE_CONFIG = {
    "test": {"n_patients": 100, "description": "Quick test (5 minutes)"},
    "medium": {"n_patients": 1000, "description": "Medium scale (1 hour, 4 batches)"},
    "large": {"n_patients": 5000, "description": "Large scale (5 hours, 20 batches)"},
    "full": {"n_patients": None, "description": "Full dataset (24-48 hours)"}
}

n_patients = MODE_CONFIG[MODE]['n_patients']
n_batches = (n_patients // BATCH_SIZE) + 1 if n_patients else "Unknown"

print(f"🚀 Running in {MODE.upper()} mode: {MODE_CONFIG[MODE]['description']}")
print(f"   Patients to process: {n_patients or 'ALL'}")
print(f"   Batch size: {BATCH_SIZE} patients")
print(f"   Estimated batches: {n_batches}")

In [ ]:
# Install dependencies
!pip install -q pandas numpy pyyaml h5py tqdm scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set up paths
import os

DRIVE_ROOT = "/content/drive/MyDrive"
MIMIC_RAW_PATH = f"{DRIVE_ROOT}/MIMIC"
PROJECT_PATH = f"{DRIVE_ROOT}/Sepsis"
OUTPUT_PATH = f"{PROJECT_PATH}/data/processed/mimic_harmonized"
CHECKPOINT_DIR = f"{OUTPUT_PATH}/checkpoints_{MODE}"

# Create directories
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"📁 MIMIC data: {MIMIC_RAW_PATH}")
print(f"📁 Output: {OUTPUT_PATH}")
print(f"📁 Checkpoints: {CHECKPOINT_DIR}")

In [ ]:
# Import modules
import sys
sys.path.append(f"{PROJECT_PATH}/src")

import pandas as pd
import numpy as np
import yaml
import h5py
import json
import pickle
from tqdm.notebook import tqdm
import logging
from datetime import datetime
import gc

from data.sofa_calculator import SOFACalculator
from data.harmonization import MIMICHarmonizer
from data.labeling import SepsisLabeler

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ Modules imported successfully")

## 2. Load Configuration

In [ ]:
# Load configuration
config_path = f"{PROJECT_PATH}/config/data_config.yaml"

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"   Variables to extract: {len(config['variable_mapping'])}")
print(f"   Prediction window: {config['sepsis_definition']['prediction_window']}")

## 3. Load Patient Cohort

In [ ]:
# Load patient cohort
print("📥 Loading patient cohort...")
icustays = pd.read_csv(f"{MIMIC_RAW_PATH}/icustays.csv.gz")
patients = pd.read_csv(f"{MIMIC_RAW_PATH}/patients.csv.gz")
admissions = pd.read_csv(f"{MIMIC_RAW_PATH}/admissions.csv.gz")

print(f"   Total ICU stays: {len(icustays)}")

# Select patient subset
if n_patients:
    icustays = icustays.head(n_patients)
    print(f"   Selected {n_patients} ICU stays for {MODE} mode")

# Merge demographics
cohort = icustays.merge(patients, on='subject_id').merge(admissions, on=['subject_id', 'hadm_id'])
cohort['intime'] = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

print(f"\n✅ Cohort ready: {len(cohort)} patients")

del icustays, patients, admissions
gc.collect()

## 4. Initialize Preprocessing Modules

In [ ]:
# Initialize harmonizer and labeler
harmonizer = MIMICHarmonizer(config_path)
labeler = SepsisLabeler(config['sepsis_definition'])

print("✅ Preprocessing modules initialized")

## 5. Batch Processing with Checkpoints

**This cell processes patients in batches and saves progress automatically.**

Features:
- Saves after each batch
- Can resume if interrupted
- Shows progress bar
- Memory efficient

In [ ]:
# ========================================
# BATCH PROCESSING WITH CHECKPOINTS
# ========================================

# Checkpoint file
checkpoint_file = f"{CHECKPOINT_DIR}/progress.json"

# Load checkpoint if exists
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        checkpoint = json.load(f)
    start_batch = checkpoint['last_completed_batch'] + 1
    print(f"📂 Resuming from checkpoint: Batch {start_batch}")
    print(f"   Previously processed: {checkpoint['patients_processed']} patients")
else:
    start_batch = 0
    checkpoint = {
        'last_completed_batch': -1,
        'patients_processed': 0,
        'sepsis_cases_found': 0,
        'batch_files': []
    }
    print("🆕 Starting fresh (no checkpoint found)")

# Split cohort into batches
total_batches = (len(cohort) // BATCH_SIZE) + (1 if len(cohort) % BATCH_SIZE > 0 else 0)
print(f"\n🔄 Processing {len(cohort)} patients in {total_batches} batches...\n")

for batch_idx in range(start_batch, total_batches):
    batch_start_time = datetime.now()
    
    # Get batch patients
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min((batch_idx + 1) * BATCH_SIZE, len(cohort))
    batch_cohort = cohort.iloc[batch_start:batch_end]
    
    print(f"\n{'='*70}")
    print(f"📦 BATCH {batch_idx + 1}/{total_batches}: Patients {batch_start+1}-{batch_end}")
    print(f"{'='*70}")
    
    # Get subject IDs for this batch
    batch_subject_ids = set(batch_cohort['subject_id'].unique())
    
    # Load data for this batch only
    print("📥 Loading clinical data for batch...")
    
    # Chartevents
    chartevents_chunks = []
    for chunk in pd.read_csv(f"{MIMIC_RAW_PATH}/chartevents.csv.gz", chunksize=100000, low_memory=False):
        filtered = chunk[chunk['subject_id'].isin(batch_subject_ids)]
        if len(filtered) > 0:
            chartevents_chunks.append(filtered)
    chartevents_batch = pd.concat(chartevents_chunks) if chartevents_chunks else pd.DataFrame()
    chartevents_batch['charttime'] = pd.to_datetime(chartevents_batch['charttime'])
    
    # Labevents
    labevents_chunks = []
    for chunk in pd.read_csv(f"{MIMIC_RAW_PATH}/labevents.csv.gz", chunksize=100000, low_memory=False):
        filtered = chunk[chunk['subject_id'].isin(batch_subject_ids)]
        if len(filtered) > 0:
            labevents_chunks.append(filtered)
    labevents_batch = pd.concat(labevents_chunks) if labevents_chunks else pd.DataFrame()
    labevents_batch['charttime'] = pd.to_datetime(labevents_batch['charttime'])
    
    # Prescriptions
    prescriptions_chunks = []
    for chunk in pd.read_csv(f"{MIMIC_RAW_PATH}/prescriptions.csv.gz", chunksize=50000, low_memory=False):
        filtered = chunk[chunk['subject_id'].isin(batch_subject_ids)]
        if len(filtered) > 0:
            prescriptions_chunks.append(filtered)
    prescriptions_batch = pd.concat(prescriptions_chunks) if prescriptions_chunks else pd.DataFrame()
    if len(prescriptions_batch) > 0:
        prescriptions_batch['starttime'] = pd.to_datetime(prescriptions_batch['starttime'])
    
    # Microbiology
    microbiology_chunks = []
    for chunk in pd.read_csv(f"{MIMIC_RAW_PATH}/microbiologyevents.csv.gz", chunksize=50000, low_memory=False):
        filtered = chunk[chunk['subject_id'].isin(batch_subject_ids)]
        if len(filtered) > 0:
            microbiology_chunks.append(filtered)
    microbiology_batch = pd.concat(microbiology_chunks) if microbiology_chunks else pd.DataFrame()
    if len(microbiology_batch) > 0:
        microbiology_batch['charttime'] = pd.to_datetime(microbiology_batch['charttime'])
    
    print(f"   Chartevents: {len(chartevents_batch):,} records")
    print(f"   Labevents: {len(labevents_batch):,} records")
    print(f"   Prescriptions: {len(prescriptions_batch):,} records")
    print(f"   Microbiology: {len(microbiology_batch):,} records")
    
    # Process patients in this batch
    batch_processed = []
    batch_sepsis_count = 0
    
    print(f"\n🔄 Processing {len(batch_cohort)} patients...")
    for idx, row in tqdm(batch_cohort.iterrows(), total=len(batch_cohort), desc=f"Batch {batch_idx+1}"):
        try:
            subject_id = row['subject_id']
            hadm_id = row['hadm_id']
            icu_intime = row['intime']
            icu_outtime = row['outtime']
            
            # Filter patient data
            patient_chart = chartevents_batch[chartevents_batch['subject_id'] == subject_id]
            patient_lab = labevents_batch[labevents_batch['subject_id'] == subject_id]
            
            if len(patient_chart) == 0 and len(patient_lab) == 0:
                continue
            
            # Harmonize
            harmonized = harmonizer.harmonize_patient(
                subject_id, patient_chart, patient_lab, icu_intime, icu_outtime
            )
            
            if len(harmonized) == 0:
                continue
            
            # Get prescriptions and microbiology
            patient_rx = prescriptions_batch[prescriptions_batch['subject_id'] == subject_id]
            patient_micro = microbiology_batch[microbiology_batch['subject_id'] == subject_id]
            
            # Label
            labeled, has_sepsis = labeler.label_patient(
                subject_id, harmonized, patient_rx, patient_micro, icu_intime, icu_outtime
            )
            
            if has_sepsis:
                batch_sepsis_count += 1
            
            # Add metadata
            labeled['hadm_id'] = hadm_id
            labeled['icu_intime'] = icu_intime
            labeled['icu_outtime'] = icu_outtime
            
            batch_processed.append(labeled)
            
        except Exception as e:
            logger.error(f"Error processing patient {subject_id}: {e}")
            continue
    
    # Save batch
    if len(batch_processed) > 0:
        batch_data = pd.concat(batch_processed, ignore_index=True)
        batch_file = f"{CHECKPOINT_DIR}/batch_{batch_idx:03d}.h5"
        batch_data.to_hdf(batch_file, key='data', mode='w', complevel=9)
        
        batch_duration = (datetime.now() - batch_start_time).total_seconds()
        
        print(f"\n✅ Batch {batch_idx + 1} complete!")
        print(f"   Processed: {len(batch_processed)} patients")
        print(f"   Sepsis cases: {batch_sepsis_count}")
        print(f"   Observations: {len(batch_data):,}")
        print(f"   Time: {batch_duration:.1f}s")
        print(f"   Saved to: {batch_file}")
        
        # Update checkpoint
        checkpoint['last_completed_batch'] = batch_idx
        checkpoint['patients_processed'] += len(batch_processed)
        checkpoint['sepsis_cases_found'] += batch_sepsis_count
        checkpoint['batch_files'].append(batch_file)
        
        with open(checkpoint_file, 'w') as f:
            json.dump(checkpoint, f, indent=2)
    
    # Clean up batch data to free memory
    del chartevents_batch, labevents_batch, prescriptions_batch, microbiology_batch
    del batch_processed, batch_data
    gc.collect()

print(f"\n\n{'='*70}")
print(f"🎉 ALL BATCHES COMPLETE!")
print(f"{'='*70}")
print(f"Total patients processed: {checkpoint['patients_processed']}")
print(f"Total sepsis cases: {checkpoint['sepsis_cases_found']}")
print(f"Sepsis prevalence: {checkpoint['sepsis_cases_found']/checkpoint['patients_processed']*100:.1f}%")
print(f"Batch files saved: {len(checkpoint['batch_files'])}")

## 6. Combine All Batches

Merge all batch files into a single dataset.

In [ ]:
# Combine all batches
print("🔗 Combining all batches...\n")

all_batches = []
for batch_file in tqdm(checkpoint['batch_files'], desc="Loading batches"):
    batch_df = pd.read_hdf(batch_file, key='data')
    all_batches.append(batch_df)
    print(f"   Loaded {batch_file}: {len(batch_df):,} observations")

all_data = pd.concat(all_batches, ignore_index=True)

print(f"\n✅ Combined dataset ready!")
print(f"   Total observations: {len(all_data):,}")
print(f"   Positive labels: {(all_data['sepsis_label']==1).sum():,}")
print(f"   Negative labels: {(all_data['sepsis_label']==0).sum():,}")

In [ ]:
# Save final combined dataset
output_file = f"{OUTPUT_PATH}/mimic_processed_{MODE}.h5"

print(f"💾 Saving final dataset to {output_file}...")
all_data.to_hdf(output_file, key='data', mode='w', complevel=9)

file_size_mb = os.path.getsize(output_file) / 1024 / 1024
print(f"✅ Data saved successfully! ({file_size_mb:.1f} MB)")

In [ ]:
# Save summary statistics
summary = {
    'mode': MODE,
    'batch_size': BATCH_SIZE,
    'n_batches': len(checkpoint['batch_files']),
    'n_patients': checkpoint['patients_processed'],
    'n_observations': len(all_data),
    'n_sepsis_cases': checkpoint['sepsis_cases_found'],
    'sepsis_prevalence': checkpoint['sepsis_cases_found'] / checkpoint['patients_processed'],
    'positive_labels': int((all_data['sepsis_label']==1).sum()),
    'negative_labels': int((all_data['sepsis_label']==0).sum()),
    'variables': list(all_data.columns),
    'processing_date': datetime.now().isoformat()
}

summary_file = f"{OUTPUT_PATH}/summary_{MODE}.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"📄 Summary saved to {summary_file}")

## 7. Data Quality Checks

In [ ]:
# Patient-level verification
print("🔍 Patient-Level Sepsis Detection:\n")
patients_with_sepsis = all_data[all_data['sepsis_label'] == 1]['subject_id'].nunique()
total_patients = all_data['subject_id'].nunique()

print(f"Unique patients with sepsis: {patients_with_sepsis}")
print(f"Total unique patients: {total_patients}")
print(f"True sepsis prevalence: {patients_with_sepsis / total_patients * 100:.1f}%")

print("\n📊 Data Quality Metrics:\n")
print("Missing Value Rates (top 10):")
missing_rates = all_data.isnull().mean().sort_values(ascending=False)
print(missing_rates[missing_rates > 0].head(10))

print("\nLabel Distribution:")
print(all_data['sepsis_label'].value_counts())

In [ ]:
# Visualizations
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Label distribution
all_data['sepsis_label'].value_counts().plot(kind='bar', ax=axes[0], color=['#3498db', '#e74c3c'])
axes[0].set_title('Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Label (0=No Sepsis, 1=Sepsis)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Time to onset
positive_data = all_data[all_data['sepsis_label'] == 1]
if 'hours_to_onset' in positive_data.columns and positive_data['hours_to_onset'].notna().any():
    positive_data['hours_to_onset'].hist(bins=50, ax=axes[1], color='#2ecc71', edgecolor='black')
    axes[1].set_title('Hours to Sepsis Onset (Positive Labels)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Hours Before Onset')
    axes[1].set_ylabel('Count')
    axes[1].axvline(x=-6, color='red', linestyle='--', label='6h prediction window')
    axes[1].legend()

plt.tight_layout()
plot_file = f"{OUTPUT_PATH}/label_distribution_{MODE}.png"
plt.savefig(plot_file, dpi=150, bbox_inches='tight')
plt.show()

print(f"📊 Plots saved to {plot_file}")

## 8. Cleanup (Optional)

Remove batch files after successful combination to save space.

In [ ]:
# OPTIONAL: Delete batch files to save space
# Uncomment the code below if you want to remove batch files

# import shutil
# 
# print("🧹 Cleaning up batch files...")
# for batch_file in checkpoint['batch_files']:
#     if os.path.exists(batch_file):
#         os.remove(batch_file)
#         print(f"   Deleted: {batch_file}")
# 
# # Remove checkpoint directory
# if os.path.exists(CHECKPOINT_DIR):
#     shutil.rmtree(CHECKPOINT_DIR)
#     print(f"   Removed checkpoint directory: {CHECKPOINT_DIR}")
# 
# print("✅ Cleanup complete!")

print("⏭️  Batch files kept for safety. Uncomment code above to delete.")

## ✅ Processing Complete!

### What you now have:
- ✅ Processed MIMIC-IV data in CinC schema
- ✅ Sepsis-3 labels generated
- ✅ Data saved in HDF5 format
- ✅ Batch files as backup

### Files created:
1. **Main dataset**: `mimic_processed_{MODE}.h5`
2. **Summary**: `summary_{MODE}.json`
3. **Plots**: `label_distribution_{MODE}.png`
4. **Checkpoints**: `checkpoints_{MODE}/batch_*.h5` (backup)

### Next steps:
1. **Scale up**: Change `MODE = "large"` or `MODE = "full"` and re-run
2. **Build model**: Use processed data for training
3. **Resume if interrupted**: Just re-run - it will continue from last batch

### Resume capability:
- If Colab disconnects, simply re-run from "Batch Processing" cell
- It will automatically resume from the last completed batch
- No progress lost!